# Exploratory Data Analysis: Inside Airbnb London (17 Sep 2026)

**A guided workbook.** Each step explains *what* to explore and *why it matters for the investment app*. The code cells contain hints as comments: you write the code yourself.

**How to use it**
1. Read the markdown cell for each step.
2. Write your code under the comments in the code cell below it.
3. Compare your result with the **Check yourself** numbers where they're given.
4. Answer the **Think about it** questions in your own words, in the empty markdown cells marked *Your notes*. These notes become your presentation talking points and documentation.

**Files** (in `data/datasets/`): `listings.csv.gz`, `calendar.csv.gz`, `reviews.csv.gz`, `neighbourhoods.geojson`

**Run this notebook from the project folder** so the relative paths work. If you open it from `notebooks/`, use `../data/datasets/` instead.

**Stuck?** Ask me for a hint on a specific step before asking for the answer: you'll learn more.

---
### Contents
0. Setup
1. Listings: first look
2. Listings: data quality
3. Listings: price
4. Listings: what's on offer (room types, property types, size)
5. Listings: where (boroughs, location)
6. Listings: reviews and activity
7. Listings: availability, minimum nights and the London 90-night rule
8. Listings: hosts
9. Listings: relationships between variables
10. Calendar
11. Reviews
12. Neighbourhoods GeoJSON
13. Bringing it together: your findings

## 0. Setup
Import the libraries you'll need and define the folder path once, so you don't repeat it.

In [ ]:
# Import pandas (as pd), numpy (as np), matplotlib.pyplot (as plt) and seaborn (as sns)
# Optional: also import json, you'll need it in step 12

# Create a variable DATA_DIR pointing to the folder "data/datasets"
# Tip: pathlib.Path makes paths work on both Windows and Mac: from pathlib import Path

# Optional: make pandas show all columns when you display a DataFrame
# Hint: pd.set_option("display.max_columns", ...)

## 1. Listings: first look
The detailed listings file is the core of the project: one row per Airbnb listing.

**Task:** load it and get a feel for its size and contents.

**Check yourself:** 91,954 rows and 90 columns.

In [ ]:
# Load listings.csv.gz into a DataFrame called listings
# pandas can read .csv.gz directly, no unzipping needed
# Hint: pd.read_csv(path, low_memory=False). low_memory=False stops a "mixed types" warning

# Print the shape (rows, columns)

In [ ]:
# Look at the first 5 rows, and at 5 random rows
# Hint: .head() and .sample(5, random_state=1)

In [ ]:
# List all column names
# Hint: .columns, or listings.columns.tolist() to see them as a list
# Try to group them in your head: listing info, host info, location, price, availability, reviews

In [ ]:
# See each column's data type and non-null count in one go
# Hint: .info()
# Look out for: which columns you'd expect to be numbers but are stored as "object" (text)?

**Think about it**
- Which columns look most useful for answering *where*, *what* and *how much* to invest?
- Which column holds the price, and what data type is it? Why is that a problem?
- What do `neighbourhood` and `neighbourhood_cleansed` contain? Which one would you use, and why?

*Your notes:*

## 2. Listings: data quality
Before analysing anything, find out what's missing or broken. The brief requires *"clear handling of missing values, outliers and data quality issues"*.

**Check yourself:** several columns are 100% empty in this snapshot (Inside Airbnb sometimes stops collecting a field).

In [ ]:
# Calculate the share of missing values in every column, sorted from most to least missing
# Hint: listings.isna().mean().sort_values(ascending=False)
# Show the top 25

In [ ]:
# Make a list of columns that are completely empty (missing share == 1)
# These can be dropped from the analysis

In [ ]:
# Check for duplicate listings
# Hint: listings["id"].duplicated().sum() should be 0

In [ ]:
# Look at the review score columns: how many listings have no review score at all?
# Compare that to how many have number_of_reviews == 0
# Are these the same listings? (Hint: compare the two boolean masks with &)

**Think about it**
- Which important columns have missing values? For each one, should you *drop the rows*, *fill a value* or *leave it missing*? Why?
- Why is filling a missing review score with 0 a bad idea?
- What does a fully empty column tell you about relying on one snapshot?

*Your notes:*

## 3. Listings: price
Price drives revenue, so it needs care. It's stored as text like `"$1,250.00"`. Inside Airbnb uses a `$` sign, but **UK prices are in pounds**.

**Check yourself:** after cleaning, median ≈ £175, mean ≈ £248, and about 35% of prices are missing. The maximum is absurd (£75,000).

In [ ]:
# Create a new numeric column price_num from the text price
# Step 1: remove everything that isn't a digit or a dot
#   Hint: listings["price"].str.replace(r"[^0-9.]", "", regex=True)
# Step 2: convert to numbers. Hint: pd.to_numeric(..., errors="coerce") turns bad values into NaN

In [ ]:
# How many prices are missing? What share is that?
# Then describe the price column: count, mean, median, min, max
# Hint: .describe(), and add percentiles=[.01, .05, .25, .5, .75, .95, .99]

In [ ]:
# Plot a histogram of price_num
# First try it as it is: what's the problem with the chart?
# Then try again limited to prices under £1,000, and/or with a log scale on the x-axis
# Hint: plt.hist(..., bins=100), or sns.histplot(data=..., x=..., log_scale=True)

In [ ]:
# Look at the 10 most expensive listings: name, room_type, bedrooms, price_num
# Do they look genuine or like errors/placeholders?
# Hint: .sort_values("price_num", ascending=False).head(10)

In [ ]:
# Decide on a valid price range (the app uses £10 to £2,000)
# Count how many listings fall outside your range

**Think about it**
- Why is the mean so much higher than the median? Which one should the app report, and why?
- Is it better to *delete* outliers or *flag* them? (Our pipeline flags them: can you explain the benefit?)
- Who has missing prices? Compare `last_review` dates for listings with and without a price.

*Your notes:*

## 4. Listings: what's on offer
An investor needs to know *what* to buy. Explore room types, property types and size.

**Check yourself:** Entire home/apt 61,074, Private room 30,471, Hotel room 207, Shared room 202.

In [ ]:
# Count listings by room_type, as numbers and as percentages
# Hint: .value_counts() and .value_counts(normalize=True)
# Make a bar chart of it

In [ ]:
# How many different property_type values are there?
# Show the 15 most common
# Why might property_type be too detailed to rank in the app?

In [ ]:
# Explore size: bedrooms, beds, bathrooms_text and accommodates
# Hint: value_counts() on bedrooms, and describe() on accommodates
# bathrooms_text is text like "1.5 shared baths": look at its most common values

In [ ]:
# Median price by room_type
# Then median price by number of bedrooms (only for Entire home/apt)
# Hint: listings.groupby("room_type")["price_num"].median()

**Think about it**
- What share of London listings are entire homes? What does that suggest about the market?
- How much more does each extra bedroom add to the median nightly price?
- How would you group bedrooms into a few buckets an investor understands?

*Your notes:*

## 5. Listings: where
London's 33 boroughs (32 + City of London) are in `neighbourhood_cleansed`.

**Check yourself:** 33 boroughs; Westminster has the most listings (10,825), then Tower Hamlets (6,723).

In [ ]:
# Count listings per borough and show all 33, sorted
# Make a horizontal bar chart (it's easier to read long names)
# Hint: .value_counts().plot(kind="barh")

In [ ]:
# Build a borough summary table with groupby + agg:
#   number of listings, median price, mean review_scores_rating, share of entire homes
# Hint: listings.groupby("neighbourhood_cleansed").agg(listings=("id","count"), median_price=("price_num","median"), ...)
# For "share of entire homes": create a True/False column first, then take its mean

In [ ]:
# Scatter plot of longitude (x) vs latitude (y), coloured by room_type
# Use a sample of ~5,000 points so it draws quickly, and small, semi-transparent dots (s=2, alpha=0.3)
# You've basically drawn a map of London without any map library

**Think about it**
- Which boroughs are most expensive? Which have the most listings? Are they the same?
- Inside Airbnb shifts locations by up to 150 m for privacy. What analysis would that affect, and what wouldn't it?
- Why is a borough a good unit for recommendations (hint: planning rules, official statistics)?

*Your notes:*

## 6. Listings: reviews and activity
Reviews are our best evidence that bookings actually happen. They also tell us whether a listing is still active.

**Check yourself:** median review score ≈ 4.83 out of 5, and about 49% of listings had no review in the 12 months before the snapshot.

In [ ]:
# Describe number_of_reviews, number_of_reviews_ltm (last 12 months) and reviews_per_month
# How many listings have 0 reviews ever?

In [ ]:
# Plot the distribution of review_scores_rating
# What do you notice about where most scores sit? Is a 4.5 "good" in this data?

In [ ]:
# Convert last_review and last_scraped to dates. Hint: pd.to_datetime
# Create days_since_review = snapshot date minus last_review (in days)
# What share of listings had a review in the last 365 days? Call these "active"

In [ ]:
# Compare the 6 review sub-scores (accuracy, cleanliness, checkin, communication, location, value)
# Which one is lowest on average? What might that suggest about what guests complain about?
# Hint: listings.filter(like="review_scores_").mean().sort_values()

**Think about it**
- Why should the scoring use *active* listings only?
- Review scores are bunched near 5. What does that mean for using them to compare areas?
- How could you use `number_of_reviews_ltm` to estimate how many nights a listing is booked? (This is the core of our occupancy model.)

*Your notes:*

## 7. Listings: availability, minimum nights and the London 90-night rule
`availability_365` = number of nights open to book in the next year. Careful: **unavailable can mean booked OR blocked by the host.**

London rule: without planning permission, an entire home can be let short term for at most **90 nights a year**.

**Check yourself:** about 31% of listings have availability_365 = 0, and about 2% require 30+ minimum nights.

In [ ]:
# Plot a histogram of availability_365
# What are the two big spikes, and what might each one mean?

In [ ]:
# Describe minimum_nights
# What share require 30+ nights? These are really medium/long-term lets

In [ ]:
# Inside Airbnb gives its own estimates: estimated_occupancy_l365d and estimated_revenue_l365d
# Describe both for entire homes
# What share of entire homes are estimated above 90 nights a year?

**Think about it**
- Why can't you treat `365 - availability_365` as nights booked?
- How should the app handle entire homes estimated above 90 nights in London?
- Would a listing with a 30-night minimum be a fair comparison for a holiday let?

*Your notes:*

## 8. Listings: hosts
Are London listings run by individuals or by professional operators? It matters for competition.

**Check yourself:** 51,348 different hosts; about 20% of hosts have more than one listing, but listings run by multi-listing hosts make up about 55% of all listings.

In [ ]:
# How many unique host_id values are there?
# Count listings per host and look at the distribution
# Who are the 10 biggest hosts by number of listings? (show host_id and count only)

In [ ]:
# Share of hosts with more than 1 listing vs share of LISTINGS owned by those hosts
# Hint: calculated_host_listings_count is already in the data

In [ ]:
# Superhosts: what share of listings have host_is_superhost == "t"?
# Compare median review score and median price for superhosts vs others

**Think about it**
- What does it mean for a new investor if more than half the supply is run by multi-listing operators?
- Is "superhost" something an investor buys, or something they earn? How would you use it?

*Your notes:*

## 9. Listings: relationships between variables
Which features go together? This guides which factors belong in the investment score.

**Check yourself:** price vs bedrooms correlation ≈ 0.23 (weaker than you might expect, because of outliers).

In [ ]:
# Pick numeric columns: price_num, accommodates, bedrooms, beds, minimum_nights, availability_365,
#   number_of_reviews_ltm, review_scores_rating, estimated_occupancy_l365d, estimated_revenue_l365d
# Make a correlation matrix and show it as a heatmap
# Hint: df[cols].corr(), then sns.heatmap(..., annot=True, fmt=".2f", cmap="coolwarm")

In [ ]:
# Redo the correlation using only valid prices (e.g. £10–£2,000)
# Does price vs bedrooms get stronger? Why?
# Try method="spearman" as well: it's less sensitive to outliers

In [ ]:
# Box plot of price by room_type (valid prices only)
# Hint: sns.boxplot(data=..., x="room_type", y="price_num")

**Think about it**
- Do higher-rated listings earn more? What does the data say?
- Why does removing outliers change correlations so much?
- Correlation isn't causation: give one example from this data where that matters.

*Your notes:*

## 10. Calendar
One row per listing per day for the next 365 days: 33.6 million rows. Load only what you need.

**Check yourself:** 33,563,890 rows; about 58% of all listing-nights are marked unavailable.

In [ ]:
# Load calendar.csv.gz with only the columns listing_id, date, available
# Hint: pd.read_csv(path, usecols=[...], parse_dates=["date"])
# If your laptop struggles with memory, read it in pieces with chunksize=5_000_000 and aggregate each chunk
# Print the shape and the first rows

In [ ]:
# What date range does it cover?
# What share of rows have available == "f"?

In [ ]:
# Share unavailable by date: group by date, take the mean of (available == "f")
# Plot it as a line chart
# Can you see weekends? Christmas? Does it drop off further in the future?

In [ ]:
# Share unavailable per listing over the next 90 days
# Plot the distribution: how many listings are almost fully unavailable?

**Think about it**
- Why does unavailability fall the further into the future you look? (Hint: people book ahead.)
- If 58% of nights are "unavailable", is London Airbnb really 58% occupied? What else could explain it?
- How would you explain this limitation to a non-technical investor in one sentence?

*Your notes:*

## 11. Reviews
Full guest comments: 2.3 million rows. This is the raw material for the AI review analysis.

**Check yourself:** 2,323,969 reviews for 71,260 listings, from Dec 2009 to Sep 2026; 237 empty comments.

In [ ]:
# Load reviews.csv.gz with columns listing_id, id, date, comments, and parse the date
# Don't print reviewer names anywhere: they're personal data
# Print the shape, the date range and the number of unique listings

In [ ]:
# Count reviews per year and plot them as a bar chart
# What happens in 2020 and 2021? What does the recovery look like?

In [ ]:
# Reviews per listing: describe the distribution
# How many reviews does the median listing have? The busiest one?

In [ ]:
# Comment length: create a column with the number of characters (or words) in each comment
# Hint: reviews["comments"].str.len(), or .str.split().str.len()
# Describe it: how long is a typical review? How many are very short (under 20 characters)?

In [ ]:
# Read 10 random comments in full
# Look for: HTML like <br/>, other languages, auto-generated messages
#   (e.g. "The host canceled this reservation...")
# Hint: for c in reviews["comments"].sample(10, random_state=1): print(c, "\n---")

In [ ]:
# Count auto-generated cancellation messages
# Hint: reviews["comments"].str.contains("canceled this reservation", case=False, na=False).sum()

**Think about it**
- Which cleaning steps would you need before sending reviews to an AI model?
- The AI budget is £1–£5. Roughly how many reviews could you afford to analyse? Why should you sample, and how (most recent? random? per listing?)
- What themes do you notice in your random sample? (Cleanliness, location, host, noise…)

*Your notes:*

## 12. Neighbourhoods GeoJSON
Borough outlines for drawing a map.

**Check yourself:** 33 features; each has a `neighbourhood` property that matches `neighbourhood_cleansed` in listings.

In [ ]:
# Open the file with json.load(open(path))
# Look at the top-level keys, how many features there are, and the properties of the first feature

In [ ]:
# Make a list of borough names from the features
# Check they all match the borough names in listings
# Hint: set(names) - set(listings["neighbourhood_cleansed"].unique()) should be empty

In [ ]:
# Optional, if you want a real map: plotly express can colour boroughs by a value
# Hint: px.choropleth_map(df, geojson=geo, locations="neighbourhood_cleansed",
#                         featureidkey="properties.neighbourhood", color="median_price", ...)
# Use your borough summary table from step 5

**Think about it**
- Which borough-level metric would be most useful to show on a map for an investor?

*Your notes:*

## 13. Bringing it together: your findings
Summarise what you found. This becomes a slide and part of your documentation.

**1. Five key insights about the London Airbnb market**
1.
2.
3.
4.
5.

**2. Data quality issues and how you'd handle each**

| Issue | How many / how big | How to handle it | Why |
|---|---|---|---|
| | | | |

**3. What this means for the investment score**
- Which factors should go into it?
- Which listings should be excluded, and why?
- Which assumptions do you need to state?

**4. Questions you still have**
-

When you're done, compare your answers with `docs/DESIGN_DECISIONS.md`. Where you disagree with a decision there, note why: it's a great discussion point for the presentation.